In [ ]:
import numpy as np
from pathlib import Path
import pickle

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


# =============================================================================
# Paths
# =============================================================================

VIT_LOGITS_PATH = (
    "/home/maria/ProjectionSort/data/"
    "google_vit-base-patch16-224_embeddings_logits.pkl"
)

HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

RESULT_DIR = Path("/home/maria/Science/results/human_vs_vit_adam_angles")

HUMAN_RESULTS_PATH = RESULT_DIR / "adam_human_labels_results.npz"
VIT_RESULTS_PATH = RESULT_DIR / "adam_vit_labels_results.npz"

OUTPATH = RESULT_DIR / "consensus_vs_disagreement_analysis.npz"


# =============================================================================
# Label loading
# =============================================================================

def load_vit_array(vit_logits_path: str) -> np.ndarray:
    path = Path(vit_logits_path)

    if not path.exists():
        raise FileNotFoundError(f"ViT logits file not found: {path}")

    if path.suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        vit = obj["natural_scenes"]

    elif path.suffix == ".npy":
        vit = np.load(path, allow_pickle=True)

    elif path.suffix in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            obj = pickle.load(f)

        if isinstance(obj, dict):
            vit = obj["natural_scenes"]
        else:
            vit = obj

    else:
        raise ValueError(f"Unsupported ViT file extension: {path.suffix}")

    vit = np.asarray(vit)

    if vit.ndim != 2 or vit.shape[1] != 1000:
        raise ValueError(f"Expected ViT logits shape (n_images, 1000), got {vit.shape}")

    return vit


def load_vit_animate_labels(vit_logits_path: str) -> np.ndarray:
    vit = load_vit_array(vit_logits_path)
    top1 = np.argmax(vit, axis=1)

    # ImageNet convention:
    # classes 0..397 are treated as animate.
    image_labels = (top1 <= 397).astype(np.int64)

    print(f"Loaded ViT logits: {vit.shape}")
    print(f"ViT label counts [inanimate, animate]: {np.bincount(image_labels, minlength=2)}")
    print("Convention: 0 = inanimate, 1 = animate")

    return image_labels


def load_human_labels(path: str) -> np.ndarray:
    labels = np.load(path, allow_pickle=True).item()["labels"]
    labels = np.asarray(labels).astype(np.int64)

    print(f"Loaded human labels: {labels.shape}")
    print("Human label counts [inanimate, animate], excluding -1:")
    print(np.bincount(labels[labels != -1], minlength=2))

    return labels


# =============================================================================
# Helpers
# =============================================================================

def sigmoid_np(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def summarize_binary_predictions(y_true, pred, score=None, name=""):
    y_true = np.asarray(y_true).astype(int)
    pred = np.asarray(pred).astype(int)

    acc = accuracy_score(y_true, pred)
    bal_acc = balanced_accuracy_score(y_true, pred)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])

    print()
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"N:                 {len(y_true)}")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred:")
    print(cm)

    auc = np.nan
    if score is not None and len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, score)
        print(f"AUC:               {auc:.4f}")

    return {
        "n": len(y_true),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "auc": auc,
        "confusion_matrix": cm,
    }


def compare_two_decoders_same_truth(
    y_true,
    human_pred,
    vit_pred,
    human_score=None,
    vit_score=None,
    name="",
):
    y_true = np.asarray(y_true).astype(int)
    human_pred = np.asarray(human_pred).astype(int)
    vit_pred = np.asarray(vit_pred).astype(int)

    human_correct = human_pred == y_true
    vit_correct = vit_pred == y_true

    d = human_correct.astype(int) - vit_correct.astype(int)

    print()
    print("#" * 80)
    print(name)
    print("#" * 80)

    print(f"N: {len(y_true)}")
    print(f"Human-trained decoder accuracy: {human_correct.mean():.4f}")
    print(f"ViT-trained decoder accuracy:   {vit_correct.mean():.4f}")
    print(f"Difference human - ViT:         {d.mean():+.4f}")
    print(f"Extra correct images:           {d.sum()}")

    human_only = np.sum((human_correct == 1) & (vit_correct == 0))
    vit_only = np.sum((human_correct == 0) & (vit_correct == 1))
    both_correct = np.sum((human_correct == 1) & (vit_correct == 1))
    both_wrong = np.sum((human_correct == 0) & (vit_correct == 0))

    print()
    print(f"Human-only correct: {human_only}")
    print(f"ViT-only correct:   {vit_only}")
    print(f"Both correct:       {both_correct}")
    print(f"Both wrong:         {both_wrong}")

    disagreement_preds = human_pred != vit_pred

    print()
    print(f"Prediction disagreement count: {np.sum(disagreement_preds)}")
    print(f"Prediction agreement count:    {np.sum(~disagreement_preds)}")

    # Paired t-test on correctness difference.
    # This is descriptive because LOO folds are not fully independent.
    if np.std(d) > 0:
        t_stat, p_two = ttest_1samp(d, popmean=0)
        p_one_human_greater = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    else:
        t_stat = np.nan
        p_two = np.nan
        p_one_human_greater = np.nan

    print()
    print("Paired t-test on image-level correctness difference")
    print(f"t-stat: {t_stat}")
    print(f"two-sided p: {p_two}")
    print(f"one-sided p, human-trained > ViT-trained: {p_one_human_greater}")

    # McNemar table:
    # rows = human-trained wrong/correct
    # cols = ViT-trained wrong/correct
    table = np.array([
        [
            np.sum((human_correct == 0) & (vit_correct == 0)),
            np.sum((human_correct == 0) & (vit_correct == 1)),
        ],
        [
            np.sum((human_correct == 1) & (vit_correct == 0)),
            np.sum((human_correct == 1) & (vit_correct == 1)),
        ],
    ])

    print()
    print("McNemar table")
    print("rows = human-trained wrong/correct")
    print("cols = ViT-trained wrong/correct")
    print(table)

    mc = mcnemar(table, exact=True)
    print(f"McNemar exact p-value: {mc.pvalue:.6f}")

    if human_score is not None and vit_score is not None:
        print()
        print("Score/logit summaries")
        print(f"Human-trained mean logit: {np.mean(human_score):+.4f}")
        print(f"ViT-trained mean logit:   {np.mean(vit_score):+.4f}")
        print(f"Human-trained median abs margin: {np.median(np.abs(human_score)):.4f}")
        print(f"ViT-trained median abs margin:   {np.median(np.abs(vit_score)):.4f}")

    return {
        "human_correct": human_correct,
        "vit_correct": vit_correct,
        "correctness_difference": d,
        "human_only_correct": human_only,
        "vit_only_correct": vit_only,
        "both_correct": both_correct,
        "both_wrong": both_wrong,
        "prediction_disagreement": disagreement_preds,
        "mcnemar_table": table,
        "mcnemar_p": mc.pvalue,
        "ttest_t": t_stat,
        "ttest_p_two_sided": p_two,
        "ttest_p_one_sided_human_greater": p_one_human_greater,
    }


def print_image_level_table(indices, human, vit, human_pred, vit_pred, human_score, vit_score, title):
    print()
    print("#" * 80)
    print(title)
    print("#" * 80)

    header = (
        "idx | human_label | vit_label | "
        "human_decoder_pred | vit_decoder_pred | "
        "human_logit | vit_logit"
    )

    print(header)
    print("-" * len(header))

    for idx in indices:
        print(
            f"{idx:03d} | "
            f"{human[idx]} | "
            f"{vit[idx]} | "
            f"{human_pred[idx]} | "
            f"{vit_pred[idx]} | "
            f"{human_score[idx]:+9.4f} | "
            f"{vit_score[idx]:+9.4f}"
        )


# =============================================================================
# Main analysis
# =============================================================================

def main():
    human_labels = load_human_labels(HUMAN_LABEL_PATH)
    vit_labels = load_vit_animate_labels(VIT_LOGITS_PATH)

    labeled_mask = human_labels != -1

    human_labels = human_labels[labeled_mask]
    vit_labels = vit_labels[labeled_mask]

    human_res = np.load(HUMAN_RESULTS_PATH, allow_pickle=True)
    vit_res = np.load(VIT_RESULTS_PATH, allow_pickle=True)

    human_y_saved = human_res["y"].astype(int)
    vit_y_saved = vit_res["y"].astype(int)

    # Safety checks.
    if not np.array_equal(human_y_saved, human_labels):
        raise ValueError("Saved human y does not match loaded human labels.")

    if not np.array_equal(vit_y_saved, vit_labels):
        raise ValueError("Saved vit y does not match loaded vit labels.")

    human_pred = human_res["adam_preds"].astype(int)
    vit_pred = vit_res["adam_preds"].astype(int)

    human_score = human_res["adam_scores"].astype(float)
    vit_score = vit_res["adam_scores"].astype(float)

    print()
    print("#" * 80)
    print("Label relationship")
    print("#" * 80)

    print("Human label counts [inanimate, animate]:", np.bincount(human_labels, minlength=2))
    print("ViT label counts   [inanimate, animate]:", np.bincount(vit_labels, minlength=2))

    label_cm = confusion_matrix(human_labels, vit_labels, labels=[0, 1])

    print()
    print("Label confusion matrix")
    print("rows = human labels [inanimate, animate]")
    print("cols = ViT labels   [inanimate, animate]")
    print(label_cm)

    consensus_mask = human_labels == vit_labels
    disagreement_mask = human_labels != vit_labels

    consensus_idx = np.where(consensus_mask)[0]
    disagreement_idx = np.where(disagreement_mask)[0]

    print()
    print(f"Consensus images:    {len(consensus_idx)}")
    print(f"Disagreement images: {len(disagreement_idx)}")

    # Because all disagreement cases in your data are expected to be human=0, vit=1.
    human0_vit1 = np.where((human_labels == 0) & (vit_labels == 1))[0]
    human1_vit0 = np.where((human_labels == 1) & (vit_labels == 0))[0]

    print()
    print("Human inanimate, ViT animate count:", len(human0_vit1))
    print("Human animate, ViT inanimate count:", len(human1_vit0))
    print("Human inanimate, ViT animate indices:", human0_vit1)
    print("Human animate, ViT inanimate indices:", human1_vit0)

    # -------------------------------------------------------------------------
    # Full-data summaries under each decoder's own training/eval label
    # -------------------------------------------------------------------------

    summarize_binary_predictions(
        human_labels,
        human_pred,
        score=human_score,
        name="Full 118 images: human-trained decoder evaluated on human labels",
    )

    summarize_binary_predictions(
        vit_labels,
        vit_pred,
        score=vit_score,
        name="Full 118 images: ViT-trained decoder evaluated on ViT labels",
    )

    # -------------------------------------------------------------------------
    # Cross-label evaluation on full 118
    # -------------------------------------------------------------------------

    summarize_binary_predictions(
        human_labels,
        vit_pred,
        score=vit_score,
        name="Full 118 images: ViT-trained decoder evaluated on HUMAN labels",
    )

    summarize_binary_predictions(
        vit_labels,
        human_pred,
        score=human_score,
        name="Full 118 images: human-trained decoder evaluated on ViT labels",
    )

    # -------------------------------------------------------------------------
    # Consensus set: shared labels, fair head-to-head
    # -------------------------------------------------------------------------

    y_consensus = human_labels[consensus_mask]

    consensus_comparison = compare_two_decoders_same_truth(
        y_true=y_consensus,
        human_pred=human_pred[consensus_mask],
        vit_pred=vit_pred[consensus_mask],
        human_score=human_score[consensus_mask],
        vit_score=vit_score[consensus_mask],
        name="Consensus 111 images: both decoders evaluated on shared labels",
    )

    summarize_binary_predictions(
        y_consensus,
        human_pred[consensus_mask],
        score=human_score[consensus_mask],
        name="Consensus images only: human-trained decoder",
    )

    summarize_binary_predictions(
        y_consensus,
        vit_pred[consensus_mask],
        score=vit_score[consensus_mask],
        name="Consensus images only: ViT-trained decoder",
    )

    # -------------------------------------------------------------------------
    # Disagreement set: the 7 relabeled images
    # -------------------------------------------------------------------------

    if len(disagreement_idx) > 0:
        print_image_level_table(
            disagreement_idx,
            human_labels,
            vit_labels,
            human_pred,
            vit_pred,
            human_score,
            vit_score,
            title="Disagreement images: human labels vs ViT labels",
        )

        # Evaluate disagreement images twice:
        # 1. against human labels
        # 2. against ViT labels
        summarize_binary_predictions(
            human_labels[disagreement_mask],
            human_pred[disagreement_mask],
            score=human_score[disagreement_mask],
            name="Disagreement images: human-trained decoder evaluated on HUMAN labels",
        )

        summarize_binary_predictions(
            human_labels[disagreement_mask],
            vit_pred[disagreement_mask],
            score=vit_score[disagreement_mask],
            name="Disagreement images: ViT-trained decoder evaluated on HUMAN labels",
        )

        summarize_binary_predictions(
            vit_labels[disagreement_mask],
            human_pred[disagreement_mask],
            score=human_score[disagreement_mask],
            name="Disagreement images: human-trained decoder evaluated on ViT labels",
        )

        summarize_binary_predictions(
            vit_labels[disagreement_mask],
            vit_pred[disagreement_mask],
            score=vit_score[disagreement_mask],
            name="Disagreement images: ViT-trained decoder evaluated on ViT labels",
        )

        print()
        print("#" * 80)
        print("Disagreement image logit behavior")
        print("#" * 80)

        print("Human-trained logits on disagreement images:")
        print(human_score[disagreement_mask])

        print()
        print("ViT-trained logits on disagreement images:")
        print(vit_score[disagreement_mask])

        print()
        print("Mean human-trained logit:", np.mean(human_score[disagreement_mask]))
        print("Mean ViT-trained logit:  ", np.mean(vit_score[disagreement_mask]))

        print()
        print("Predicted animate rate on disagreement images")
        print("Human-trained decoder:", np.mean(human_pred[disagreement_mask] == 1))
        print("ViT-trained decoder:  ", np.mean(vit_pred[disagreement_mask] == 1))

    # -------------------------------------------------------------------------
    # Save all useful arrays
    # -------------------------------------------------------------------------

    np.savez_compressed(
        OUTPATH,
        human_labels=human_labels,
        vit_labels=vit_labels,
        human_pred=human_pred,
        vit_pred=vit_pred,
        human_score=human_score,
        vit_score=vit_score,
        consensus_mask=consensus_mask,
        disagreement_mask=disagreement_mask,
        consensus_indices=consensus_idx,
        disagreement_indices=disagreement_idx,
        label_confusion_matrix=label_cm,
        consensus_human_correct=consensus_comparison["human_correct"],
        consensus_vit_correct=consensus_comparison["vit_correct"],
        consensus_correctness_difference=consensus_comparison["correctness_difference"],
        consensus_mcnemar_table=consensus_comparison["mcnemar_table"],
        consensus_mcnemar_p=consensus_comparison["mcnemar_p"],
        consensus_prediction_disagreement=consensus_comparison["prediction_disagreement"],
    )

    print()
    print("Saved:", OUTPATH)


if __name__ == "__main__":
    main()

Loaded human labels: (118,)
Human label counts [inanimate, animate], excluding -1:
[62 56]
Loaded ViT logits: (118, 1000)
ViT label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate

################################################################################
Label relationship
################################################################################
Human label counts [inanimate, animate]: [62 56]
ViT label counts   [inanimate, animate]: [55 63]

Label confusion matrix
rows = human labels [inanimate, animate]
cols = ViT labels   [inanimate, animate]
[[55  7]
 [ 0 56]]

Consensus images:    111
Disagreement images: 7

Human inanimate, ViT animate count: 7
Human animate, ViT inanimate count: 0
Human inanimate, ViT animate indices: [ 64  70  76  78  93 107 110]
Human animate, ViT inanimate indices: []

Full 118 images: human-trained decoder evaluated on human labels
N:                 118
Accuracy:          0.7203
Balanced accuracy: 0.7183
Confusion m

/home/maria/global_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2776: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/maria/global_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2776: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/maria/global_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2776: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/maria/global_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2776: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [3]:
import csv
import pickle
import shutil
from pathlib import Path

import numpy as np
from sklearn.metrics import confusion_matrix


# =============================================================================
# Paths
# =============================================================================

VIT_LOGITS_PATH = (
    "/home/maria/ProjectionSort/data/"
    "google_vit-base-patch16-224_embeddings_logits.pkl"
)

HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"
IMAGE_DIR = Path("/home/maria/Science/data/images")

RESULT_DIR = Path("/home/maria/Science/results/human_vs_vit_adam_angles")

HUMAN_RESULTS_PATH = RESULT_DIR / "adam_human_labels_results.npz"
VIT_RESULTS_PATH = RESULT_DIR / "adam_vit_labels_results.npz"

OUTDIR = RESULT_DIR / "consensus_case_images"
OUTDIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Loading labels
# =============================================================================

def load_vit_array(vit_logits_path: str) -> np.ndarray:
    path = Path(vit_logits_path)

    if not path.exists():
        raise FileNotFoundError(f"ViT logits file not found: {path}")

    if path.suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        if "natural_scenes" not in obj:
            raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
        vit = obj["natural_scenes"]

    elif path.suffix == ".npy":
        vit = np.load(path, allow_pickle=True)

    elif path.suffix in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            obj = pickle.load(f)

        if isinstance(obj, dict):
            if "natural_scenes" not in obj:
                raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
            vit = obj["natural_scenes"]
        else:
            vit = obj

    else:
        raise ValueError(f"Unsupported ViT file extension: {path.suffix}")

    vit = np.asarray(vit)

    if vit.ndim != 2 or vit.shape[1] != 1000:
        raise ValueError(f"Expected ViT logits shape (n_images, 1000), got {vit.shape}")

    return vit


def load_vit_animate_labels(vit_logits_path: str) -> np.ndarray:
    vit = load_vit_array(vit_logits_path)
    top1 = np.argmax(vit, axis=1)

    # ImageNet convention:
    # classes 0..397 are treated as animate.
    image_labels = (top1 <= 397).astype(np.int64)

    print(f"Loaded ViT logits: {vit.shape}")
    print(f"ViT label counts [inanimate, animate]: {np.bincount(image_labels, minlength=2)}")
    print("Convention: 0 = inanimate, 1 = animate")

    return image_labels


def load_human_labels(path: str) -> np.ndarray:
    labels = np.load(path, allow_pickle=True).item()["labels"]
    labels = np.asarray(labels).astype(np.int64)

    print(f"Loaded human labels: {labels.shape}")
    print("Human label counts [inanimate, animate], excluding -1:")
    print(np.bincount(labels[labels != -1], minlength=2))

    return labels


# =============================================================================
# Helpers
# =============================================================================

def image_path_from_original_index(idx: int) -> Path:
    return IMAGE_DIR / f"scene_{idx:03d}.png"


def write_case_images_and_csv(
    case_name,
    global_indices,
    human_labels_full,
    vit_labels_full,
    human_pred_full,
    vit_pred_full,
    human_score_full,
    vit_score_full,
):
    """
    Writes:
      1. A CSV table describing selected images.
      2. Copies scene_XXX.png files into a case folder.

    The global_indices must refer to original image indices:
        64 -> /home/maria/Science/data/images/scene_064.png
    """
    case_dir = OUTDIR / case_name
    case_dir.mkdir(parents=True, exist_ok=True)

    csv_path = case_dir / f"{case_name}.csv"

    rows = []

    for idx in global_indices:
        idx = int(idx)
        src = image_path_from_original_index(idx)
        dst = case_dir / src.name

        if src.exists():
            shutil.copy2(src, dst)
            copied_path = str(dst)
        else:
            copied_path = ""
            print(f"[WARNING] Image file not found: {src}")

        rows.append({
            "original_index": idx,
            "image_name": src.name,
            "image_path": str(src),
            "copied_path": copied_path,
            "human_label": int(human_labels_full[idx]),
            "vit_label": int(vit_labels_full[idx]),
            "human_decoder_pred": int(human_pred_full[idx]),
            "vit_decoder_pred": int(vit_pred_full[idx]),
            "human_logit": float(human_score_full[idx]),
            "vit_logit": float(vit_score_full[idx]),
            "human_decoder_correct_under_human_label": int(human_pred_full[idx] == human_labels_full[idx]),
            "vit_decoder_correct_under_human_label": int(vit_pred_full[idx] == human_labels_full[idx]),
            "human_decoder_correct_under_vit_label": int(human_pred_full[idx] == vit_labels_full[idx]),
            "vit_decoder_correct_under_vit_label": int(vit_pred_full[idx] == vit_labels_full[idx]),
        })

    fieldnames = [
        "original_index",
        "image_name",
        "image_path",
        "copied_path",
        "human_label",
        "vit_label",
        "human_decoder_pred",
        "vit_decoder_pred",
        "human_logit",
        "vit_logit",
        "human_decoder_correct_under_human_label",
        "vit_decoder_correct_under_human_label",
        "human_decoder_correct_under_vit_label",
        "vit_decoder_correct_under_vit_label",
    ]

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print()
    print("=" * 80)
    print(f"Wrote case set: {case_name}")
    print("=" * 80)
    print(f"N: {len(global_indices)}")
    print(f"CSV: {csv_path}")
    print(f"Copied images to: {case_dir}")
    print("Indices:")
    print(np.asarray(global_indices, dtype=int))

    return csv_path, case_dir


def print_case_table(
    case_name,
    global_indices,
    human_labels_full,
    vit_labels_full,
    human_pred_full,
    vit_pred_full,
    human_score_full,
    vit_score_full,
):
    print()
    print("#" * 80)
    print(case_name)
    print("#" * 80)

    header = (
        "idx | human_label | vit_label | "
        "human_pred | vit_pred | "
        "human_logit | vit_logit | image"
    )
    print(header)
    print("-" * len(header))

    for idx in global_indices:
        idx = int(idx)
        print(
            f"{idx:03d} | "
            f"{human_labels_full[idx]} | "
            f"{vit_labels_full[idx]} | "
            f"{human_pred_full[idx]} | "
            f"{vit_pred_full[idx]} | "
            f"{human_score_full[idx]:+9.4f} | "
            f"{vit_score_full[idx]:+9.4f} | "
            f"scene_{idx:03d}.png"
        )


# =============================================================================
# Main
# =============================================================================

def main():
    # -------------------------------------------------------------------------
    # Load labels
    # -------------------------------------------------------------------------

    human_labels = load_human_labels(HUMAN_LABEL_PATH)
    vit_labels = load_vit_animate_labels(VIT_LOGITS_PATH)

    labeled_mask = human_labels != -1
    original_indices_after_label_mask = np.where(labeled_mask)[0]

    human_labels_labeled = human_labels[labeled_mask]
    vit_labels_labeled = vit_labels[labeled_mask]

    # -------------------------------------------------------------------------
    # Load saved decoder results
    # -------------------------------------------------------------------------

    human_res = np.load(HUMAN_RESULTS_PATH, allow_pickle=True)
    vit_res = np.load(VIT_RESULTS_PATH, allow_pickle=True)

    human_y_saved = human_res["y"].astype(int)
    vit_y_saved = vit_res["y"].astype(int)

    if not np.array_equal(human_y_saved, human_labels_labeled):
        raise ValueError("Saved human y does not match loaded human labels.")

    if not np.array_equal(vit_y_saved, vit_labels_labeled):
        raise ValueError("Saved ViT y does not match loaded ViT labels.")

    human_pred_labeled = human_res["adam_preds"].astype(int)
    vit_pred_labeled = vit_res["adam_preds"].astype(int)

    human_score_labeled = human_res["adam_scores"].astype(float)
    vit_score_labeled = vit_res["adam_scores"].astype(float)

    # Re-expand predictions back to full 118-index coordinate system.
    # This makes original index -> scene_XXX.png safe.
    n_full = len(human_labels)

    human_pred_full = np.full(n_full, -1, dtype=int)
    vit_pred_full = np.full(n_full, -1, dtype=int)

    human_score_full = np.full(n_full, np.nan, dtype=float)
    vit_score_full = np.full(n_full, np.nan, dtype=float)

    human_pred_full[labeled_mask] = human_pred_labeled
    vit_pred_full[labeled_mask] = vit_pred_labeled

    human_score_full[labeled_mask] = human_score_labeled
    vit_score_full[labeled_mask] = vit_score_labeled

    # -------------------------------------------------------------------------
    # Relationship between human and ViT labels
    # -------------------------------------------------------------------------

    print()
    print("#" * 80)
    print("Label relationship")
    print("#" * 80)

    print("Human label counts [inanimate, animate]:",
          np.bincount(human_labels_labeled, minlength=2))
    print("ViT label counts   [inanimate, animate]:",
          np.bincount(vit_labels_labeled, minlength=2))

    label_cm = confusion_matrix(human_labels_labeled, vit_labels_labeled, labels=[0, 1])

    print()
    print("Label confusion matrix")
    print("rows = human labels [inanimate, animate]")
    print("cols = ViT labels   [inanimate, animate]")
    print(label_cm)

    consensus_mask_labeled = human_labels_labeled == vit_labels_labeled
    disagreement_mask_labeled = human_labels_labeled != vit_labels_labeled

    consensus_global_idx = original_indices_after_label_mask[consensus_mask_labeled]
    disagreement_global_idx = original_indices_after_label_mask[disagreement_mask_labeled]

    print()
    print(f"Consensus images:    {len(consensus_global_idx)}")
    print(f"Disagreement images: {len(disagreement_global_idx)}")

    print("Disagreement indices:", disagreement_global_idx)

    # -------------------------------------------------------------------------
    # Case sets on consensus images
    # -------------------------------------------------------------------------

    shared_truth = human_labels[consensus_global_idx]

    human_pred_consensus = human_pred_full[consensus_global_idx]
    vit_pred_consensus = vit_pred_full[consensus_global_idx]

    human_correct_consensus = human_pred_consensus == shared_truth
    vit_correct_consensus = vit_pred_consensus == shared_truth

    human_only_correct_idx = consensus_global_idx[
        (human_correct_consensus == 1) & (vit_correct_consensus == 0)
    ]

    vit_only_correct_idx = consensus_global_idx[
        (human_correct_consensus == 0) & (vit_correct_consensus == 1)
    ]

    both_wrong_idx = consensus_global_idx[
        (human_correct_consensus == 0) & (vit_correct_consensus == 0)
    ]

    both_correct_idx = consensus_global_idx[
        (human_correct_consensus == 1) & (vit_correct_consensus == 1)
    ]

    print()
    print("#" * 80)
    print("Consensus case counts")
    print("#" * 80)

    print("Human-trained correct, ViT-trained wrong:", len(human_only_correct_idx))
    print("ViT-trained correct, human-trained wrong:", len(vit_only_correct_idx))
    print("Both wrong:", len(both_wrong_idx))
    print("Both correct:", len(both_correct_idx))
    print("Net human advantage:",
          len(human_only_correct_idx) - len(vit_only_correct_idx))

    # -------------------------------------------------------------------------
    # Print tables
    # -------------------------------------------------------------------------

    print_case_table(
        "Human-trained correct, ViT-trained wrong on CONSENSUS images",
        human_only_correct_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    print_case_table(
        "ViT-trained correct, human-trained wrong on CONSENSUS images",
        vit_only_correct_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    print_case_table(
        "Both decoders wrong on CONSENSUS images",
        both_wrong_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    print_case_table(
        "The 7 human/ViT DISAGREEMENT images",
        disagreement_global_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    # -------------------------------------------------------------------------
    # Write CSVs and copy image files
    # -------------------------------------------------------------------------

    write_case_images_and_csv(
        "human_correct_vit_wrong_consensus",
        human_only_correct_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    write_case_images_and_csv(
        "vit_correct_human_wrong_consensus",
        vit_only_correct_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    write_case_images_and_csv(
        "both_wrong_consensus",
        both_wrong_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    write_case_images_and_csv(
        "human_vit_label_disagreement_images",
        disagreement_global_idx,
        human_labels,
        vit_labels,
        human_pred_full,
        vit_pred_full,
        human_score_full,
        vit_score_full,
    )

    # -------------------------------------------------------------------------
    # Save indices as npz too
    # -------------------------------------------------------------------------

    npz_path = OUTDIR / "consensus_case_image_indices.npz"

    np.savez_compressed(
        npz_path,
        human_correct_vit_wrong_consensus=human_only_correct_idx,
        vit_correct_human_wrong_consensus=vit_only_correct_idx,
        both_wrong_consensus=both_wrong_idx,
        both_correct_consensus=both_correct_idx,
        disagreement_indices=disagreement_global_idx,
        consensus_indices=consensus_global_idx,
    )

    print()
    print("=" * 80)
    print("Saved index arrays")
    print("=" * 80)
    print(npz_path)

    print()
    print("Done. Main folder:")
    print(OUTDIR)


if __name__ == "__main__":
    main()

Loaded human labels: (118,)
Human label counts [inanimate, animate], excluding -1:
[62 56]
Loaded ViT logits: (118, 1000)
ViT label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate

################################################################################
Label relationship
################################################################################
Human label counts [inanimate, animate]: [62 56]
ViT label counts   [inanimate, animate]: [55 63]

Label confusion matrix
rows = human labels [inanimate, animate]
cols = ViT labels   [inanimate, animate]
[[55  7]
 [ 0 56]]

Consensus images:    111
Disagreement images: 7
Disagreement indices: [ 64  70  76  78  93 107 110]

################################################################################
Consensus case counts
################################################################################
Human-trained correct, ViT-trained wrong: 7
ViT-trained correct, human-trained wrong: 4
Both wrong: 2

In [5]:
import numpy as np

def hoeffding_radius_binary(n, delta=0.05):
    return np.sqrt(np.log(2 / delta) / (2 * n))

def hoeffding_radius_paired_difference(n, delta=0.05):
    # d_i in {-1, 0, +1}, range length = 2
    return np.sqrt(2 * np.log(2 / delta) / n)

for n in [7, 111, 118]:
    print(f"n={n}")
    print("binary accuracy radius:", hoeffding_radius_binary(n))
    print("paired difference radius:", hoeffding_radius_paired_difference(n))
    print()

n = 118

obs_human_acc = 0.7203
obs_vit_acc = 0.6949
obs_diff = obs_human_acc - obs_vit_acc

r_acc = hoeffding_radius_binary(n)
r_diff = hoeffding_radius_paired_difference(n)

print("Human-trained accuracy on human ground truth Hoeffding interval:",
      obs_human_acc - r_acc, obs_human_acc + r_acc)

print("ViT-trained accuracy on human ground truth Hoeffding interval:",
      obs_vit_acc - r_acc, obs_vit_acc + r_acc)

print("Paired difference Hoeffding interval:",
      obs_diff - r_diff, obs_diff + r_diff)

n=7
binary accuracy radius: 0.5133141236899359
paired difference radius: 1.0266282473798718

n=111
binary accuracy radius: 0.12890529127087974
paired difference radius: 0.2578105825417595

n=118
binary accuracy radius: 0.12502337839200545
paired difference radius: 0.2500467567840109

Human-trained accuracy on human ground truth Hoeffding interval: 0.5952766216079945 0.8453233783920056
ViT-trained accuracy on human ground truth Hoeffding interval: 0.5698766216079945 0.8199233783920055
Paired difference Hoeffding interval: -0.2246467567840108 0.275446756784011
